# 00: Настройка окружения
Монтирование диска, установка пакетов, инициализация конфига и логгера.

In [ ]:
import sys
from pathlib import Path
from google.colab import drive

# Монтирование Google Drive
drive.mount('/content/drive')

# Определяем корневые пути
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Projects/quantum_workspace")
SRC_PATH = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "configs"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

# Создаём папки, если их нет
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
(OUTPUT_PATH / "models").mkdir(exist_ok=True)
(OUTPUT_PATH / "logs").mkdir(exist_ok=True)

# Добавляем src в PYTHONPATH
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path includes src: {str(SRC_PATH) in sys.path}")

In [ ]:
import subprocess
import sys

def install_requirements(requirements_path: Path):
    """Установка пакетов из requirements.txt."""
    if not requirements_path.exists():
        print(f"File {requirements_path} not found, skipping installation.")
        return
    
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements_path), "--quiet"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print(f"Error installing packages:\n{result.stderr}")
    else:
        print("All packages installed successfully.")

# Установка пакетов
install_requirements(PROJECT_ROOT / "requirements.txt")

In [ ]:
from core.config_loader import init_config, get_config
from core.logger import setup_logging

# Инициализируем конфиг
init_config(PROJECT_ROOT / "configs" / "config.yaml")
cfg = get_config()

# Настраиваем логирование
setup_logging(
    log_level=cfg.logging.level,
    log_format=cfg.logging.format,
    log_file=PROJECT_ROOT / cfg.logging.file
)

print(f"Environment: {cfg.environment}")
print(f"Backend (QCloud chip): {cfg.backends.qcloud.chip}")
print(f"Backend (Octillion chip): {cfg.backends.octillion.chip}")
print(f"Logging level: {cfg.logging.level}")
print("Configuration and logging initialized.")

In [ ]:
import os
from google.colab import userdata

def load_secrets():
    """Загружает API-ключи из colab.userdata в переменные окружения."""
    secrets = {
        'QPANDA_QCLOUD_API_KEY': 'QCLOUD_API_KEY',
        'OCTILLION_TOKEN': 'OCTILLION_TOKEN',
        'WANDB_API_KEY': 'WANDB_API_KEY',
    }
    
    for env_var, secret_name in secrets.items():
        try:
            os.environ[env_var] = userdata.get(secret_name)
            print(f"✓ {env_var} loaded from userdata")
        except Exception:
            print(f"⚠ {secret_name} not found in userdata. Set it via left panel → Secrets.")

load_secrets()

In [ ]:
import sys

def reload_project_modules():
    """Перезагружает все модули проекта после редактирования .py файлов."""
    modules_order = ['core', 'utils', 'pipelines']
    
    for prefix in modules_order:
        to_reload = [
            name for name in sys.modules
            if name.startswith(prefix) or name.startswith(f"src.{prefix}")
        ]
        for name in sorted(to_reload, reverse=True):
            if name in sys.modules:
                del sys.modules[name]
    
    print("Project modules cache cleared. Ready for fresh imports.")

reload_project_modules()